# SAM 3 Plant Analysis — VitroVision v2วิเคราะห์ต้นพืชเพาะเลี้ยงเนื้อเยื่อ (in vitro) จากภาพขวดแก้ว ด้วย **SAM 3 (`facebook/sam3`)** — instance segmentation จาก text prompt แล้วดึงพารามิเตอร์เชิงปริมาณ (จำนวน, พื้นที่, coverage, ขนาด bbox, ความเขียว) เพื่อทดสอบว่าตัวไหนวัดการเจริญของต้นได้ดี## วิธีใช้ (ทำครั้งเดียวตอนเริ่ม)1. **เลือก GPU runtime:** เมนู `Runtime → Change runtime type` → เลือก **T4 GPU** → Save2. **Login Hugging Face:** รัน cell ที่ 3 แล้วใส่ token ที่มีสิทธิ์เข้าถึงโมเดล `facebook/sam3` (เป็น **gated model** — ต้องขอ access ที่หน้าโมเดลก่อน) หา token ได้ที่ https://huggingface.co/settings/tokens3. รัน cell ถัดไปตามลำดับ แล้วอัปโหลดภาพขวดเพาะเลี้ยงเนื้อเยื่อจริง (รองรับหลายไฟล์ในครั้งเดียว)> ⚠️ ต้องใช้ GPU runtime เท่านั้น — `facebook/sam3` ไม่รองรับ CPU/ONNX และห้ามเปลี่ยนเป็น `facebook/sam3.1` (ยังไม่มี transformers integration)

In [ ]:
# ติดตั้ง dependency ที่ต้องใช้ทั้งหมด!pip install -q transformers torch torchvision opencv-python pillow matplotlib pandas numpy huggingface_hub

In [ ]:
# Login Hugging Face — ต้องใช้ token ที่มี access ถึง facebook/sam3 (gated model)# วิธีหา token: https://huggingface.co/settings/tokens → New token → สิทธิ์ readfrom huggingface_hub import notebook_loginnotebook_login()

In [ ]:
# ===== คอนฟิก — แก้ได้ตามต้องการ =====# คำ (prompt) ที่ใช้สั่งให้ SAM3 แยกส่วนต้นพืช เช่น ["plant", "leaf", "shoot"]# ลองแก้เพิ่ม/ลดได้ เช่น ["plant"], ["leaf", "stem", "roots"]PROMPTS = ["plant", "leaf", "shoot"]# ค่า threshold ของ confidence ในการคัด maskSCORE_THRESHOLD = 0.5# ถ้ารู้สเกลจริง: 1 px = กี่ cm (เช่น 0.02) → จะคำนวณ total_area_cm2 ให้# ถ้าใส่ None จะคืนพื้นที่เป็น px² อย่างเดียวPIXEL_TO_CM = None

In [ ]:
# ตรวจ device + โหลดโมเดล SAM 3import timeimport torchdevice = "cuda" if torch.cuda.is_available() else "cpu"print(f"device: {device}")if device == "cpu":    print("⚠️ ตรวจไม่พบ GPU — ต้องเปลี่ยน runtime เป็น T4 GPU (Runtime → Change runtime type) แล้วรันใหม่")else:    print(f"GPU: {torch.cuda.get_device_name(0)}")from transformers import Sam3Processor, Sam3Modelstart = time.time()model = Sam3Model.from_pretrained("facebook/sam3").to(device)processor = Sam3Processor.from_pretrained("facebook/sam3")print(f"โหลดโมเดลเสร็จใน {time.time() - start:.1f} วินาที")

In [ ]:
# อัปโหลดภาพขวดเพาะเลี้ยงเนื้อเยื่อ (รองรับหลายไฟล์ในครั้งเดียว)from google.colab import filesfrom PIL import Imageimport matplotlib.pyplot as pltimport iouploaded = files.upload()images = {}for name, data in uploaded.items():    img = Image.open(io.BytesIO(data)).convert("RGB")   # แปลงเป็น RGB เสมอ    images[name] = img    print(f"โหลดแล้ว: {name} — ขนาด {img.size}")if not images:    raise RuntimeError("ยังไม่มีการอัปโหลดภาพ — รัน cell นี้ใหม่แล้วเลือกไฟล์ภาพ")# แสดงภาพทั้งหมดที่อัปโหลดn = len(images)fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))if n == 1:    axes = [axes]for ax, (name, img) in zip(axes, images.items()):    ax.imshow(img)    ax.set_title(name)    ax.axis("off")plt.tight_layout()plt.show()

In [ ]:
# ฟังก์ชันดึงพารามิเตอร์เชิงปริมาณจาก masks — ใช้ numpy/cv2/PIL ล้วน ไม่มี dependency เพิ่มimport numpy as npimport cv2def analyze_plant(image, masks, scores):    rgb = np.array(image)                 # H×W×3 (0-255)    h, w = rgb.shape[:2]    # รวมทุก mask เป็น union เดียว    union = np.zeros((h, w), dtype=bool)    for m in masks:        union |= m    # พื้นที่รวม (px²)    total_area_px = int(union.sum())    # พื้นที่รวมเป็น cm² (เฉพาะเมื่อใส่ PIXEL_TO_CM)    total_area_cm2 = None    if PIXEL_TO_CM:        total_area_cm2 = round(total_area_px * (PIXEL_TO_CM ** 2), 4)    # สัดส่วนพื้นที่ที่ปกคลุมเทียบทั้งภาพ (0-1)    coverage_ratio = round(total_area_px / (h * w), 6)    # bounding box รวมของต้น (proxy ความสูง/ความกว้าง)    ys, xs = np.where(union)    if len(ys) > 0:        bbox_h_px = int(ys.max() - ys.min())        bbox_w_px = int(xs.max() - xs.min())    else:        bbox_h_px = 0        bbox_w_px = 0    # ความเขียว: ค่าเฉลี่ย G/(R+G+B) เฉพาะพิกเซลใน mask    if total_area_px > 0:        rgb_f = rgb.astype(np.float32)        r = rgb_f[..., 0][union]        g = rgb_f[..., 1][union]        b = rgb_f[..., 2][union]        denom = r + g + b        greenness = float(np.mean(g / np.maximum(denom, 1)))    else:        greenness = 0.0    # confidence เฉลี่ยของ masks    mean_score = float(np.mean(scores)) if len(scores) > 0 else 0.0    return {        "count": len(masks),        "total_area_px": total_area_px,        "total_area_cm2": total_area_cm2,        "coverage_ratio": coverage_ratio,        "bbox_h_px": bbox_h_px,        "bbox_w_px": bbox_w_px,        "greenness": round(greenness, 4),        "mean_score": round(mean_score, 4),    }

In [ ]:
# รันวิเคราะห์ทุกภาพ × ทุก prompt และรวมผลเป็น DataFrameimport pandas as pdrows = []all_results = {}   # {ชื่อภาพ: {prompt: {"masks", "scores", "params"}}} — เก็บไว้ให้ cell ถัดไปใช้total_jobs = len(images) * len(PROMPTS)job = 0for img_name, img in images.items():    all_results[img_name] = {}    for prompt in PROMPTS:        job += 1        print(f"Processing {img_name} / prompt='{prompt}' [{job}/{total_jobs}]...")        inputs = processor(images=img, text=prompt, return_tensors="pt").to(device)        with torch.no_grad():            outputs = model(**inputs)        results = processor.post_process_instance_segmentation(            outputs, threshold=SCORE_THRESHOLD, mask_threshold=0.5,            target_sizes=inputs.get("original_sizes").tolist()        )[0]        masks = results["masks"].cpu().numpy().astype(bool)   # N×H×W ขนาดเดียวกับภาพต้นฉบับ        scores = results.get("scores")        scores = scores.cpu().numpy() if scores is not None else np.array([])        params = analyze_plant(img, masks, scores)        all_results[img_name][prompt] = {"masks": masks, "scores": scores, "params": params}        rows.append({"image": img_name, "prompt": prompt, **params})df = pd.DataFrame(rows)print(f"รวม {len(df)} แถว ({len(images)} ภาพ × {len(PROMPTS)} prompts)\n")# แสดงผล: head + full print เมื่อจำนวนแถวไม่เยอะif len(df) <= 10:    print(df.to_string(index=False))else:    print(df.head().to_string(index=False))    print(f"\n... รวมทั้งหมด {len(df)} แถว")

In [ ]:
# แสดง overlay: contour สีเขียวขีดรอบทุก mask เทียบกับภาพต้นฉบับ + ตารางพารามิเตอร์ของภาพนั้นfrom IPython.display import displaydef draw_overlay(img, masks):    img_rgb = np.array(img).copy()    for m in masks:        mask_u8 = (m.astype(np.uint8)) * 255        contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)        cv2.drawContours(img_rgb, contours, -1, (0, 200, 0), 2)    return img_rgbfor img_name, img in images.items():    n_panels = 1 + len(PROMPTS)    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 5))    axes[0].imshow(img)    axes[0].set_title("ต้นฉบับ")    axes[0].axis("off")    for i, prompt in enumerate(PROMPTS):        entry = all_results[img_name][prompt]        overlay = draw_overlay(img, entry["masks"])        axes[i + 1].imshow(overlay)        axes[i + 1].set_title(f"prompt='{prompt}' — เจอ {entry['params']['count']} mask")        axes[i + 1].axis("off")    plt.suptitle(img_name, fontsize=14)    plt.tight_layout()    plt.show()    # ตารางพารามิเตอร์ของภาพนี้    display(df[df["image"] == img_name])

In [ ]:
# Export CSV + สรุปค่าเฉลี่ยของแต่ละคอลัมน์แยกตาม promptfrom google.colab import files as colab_filescsv_path = "sam3_plant_analysis_results.csv"df.to_csv(csv_path, index=False, encoding="utf-8-sig")   # utf-8-sig ให้ Excel เปิดหัวตารางภาษาไทยได้ถูกต้องprint(f"บันทึก CSV แล้ว: {csv_path}")colab_files.download(csv_path)# สรุป: ค่าเฉลี่ยของแต่ละคอลัมน์แยกตาม promptnum_cols = ["count", "total_area_px", "total_area_cm2", "coverage_ratio",            "bbox_h_px", "bbox_w_px", "greenness", "mean_score"]summary = df.groupby("prompt")[num_cols].mean()print("\n=== สรุปค่าเฉลี่ยแยกตาม prompt ===")print(summary.to_string())